In [13]:
import pandas as pd
import numpy as np
import joblib

In [14]:
df = pd.read_csv("../../Datasets_For_Model_Training/Final_Merged_Dataset.csv")
df.head()

,Date,Electricity_Requirement,Humidity,Rainfall,Electricity_Supply,Solar_Irradiance,Temperature
0,2015-04-01,8361.0,70.07,130.566857,8112.0,166.53,28.57
1,2015-05-01,8381.0,77.22,160.792286,8165.0,155.03,27.95
2,2015-06-01,8302.0,77.55,98.240286,8257.0,160.31,27.31
3,2015-07-01,8953.0,73.18,37.804286,8901.0,166.41,27.68
4,2015-08-01,8535.0,72.58,63.944286,8531.0,167.12,27.85


In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 129 entries, 0 to 128
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Date                     129 non-null    object 
 1   Electricity_Requirement  129 non-null    float64
 2   Humidity                 129 non-null    float64
 3   Rainfall                 129 non-null    float64
 4   Electricity_Supply       129 non-null    float64
 5   Solar_Irradiance         129 non-null    float64
 6   Temperature              129 non-null    float64
dtypes: float64(6), object(1)
memory usage: 7.2+ KB


In [16]:
df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d")

In [17]:
# extracting year and month
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month

In [18]:
import numpy as np

# Cyclical Encoding of Month
df["Month_sin"] = np.sin(2 * np.pi * df["Month"] / 12)
df["Month_cos"] = np.cos(2 * np.pi * df["Month"] / 12)

# Verify the encoding
print(
    df[["Month", "Month_sin", "Month_cos"]]
    .drop_duplicates()
    .sort_values("Month")
)

    Month     Month_sin     Month_cos
9       1  5.000000e-01  8.660254e-01
10      2  8.660254e-01  5.000000e-01
11      3  1.000000e+00  6.123234e-17
0       4  8.660254e-01 -5.000000e-01
1       5  5.000000e-01 -8.660254e-01
2       6  1.224647e-16 -1.000000e+00
3       7 -5.000000e-01 -8.660254e-01
4       8 -8.660254e-01 -5.000000e-01
5       9 -1.000000e+00 -1.836970e-16
6      10 -8.660254e-01  5.000000e-01
7      11 -5.000000e-01  8.660254e-01
8      12 -2.449294e-16  1.000000e+00


In [19]:
print(df["Date"].head(15))

0    2015-04-01
1    2015-05-01
2    2015-06-01
3    2015-07-01
4    2015-08-01
5    2015-09-01
6    2015-10-01
7    2015-11-01
8    2015-12-01
9    2016-01-01
10   2016-02-01
11   2016-03-01
12   2016-04-01
13   2016-05-01
14   2016-06-01
Name: Date, dtype: datetime64[ns]


In [20]:
df.dropna(inplace=True)

In [21]:
df.head()

,Date,Electricity_Requirement,Humidity,Rainfall,Electricity_Supply,Solar_Irradiance,Temperature,Year,Month,Month_sin,Month_cos
0,2015-04-01,8361.0,70.07,130.566857,8112.0,166.53,28.57,2015,4,8.660254e-01,-0.500000
1,2015-05-01,8381.0,77.22,160.792286,8165.0,155.03,27.95,2015,5,5.000000e-01,-0.866025
2,2015-06-01,8302.0,77.55,98.240286,8257.0,160.31,27.31,2015,6,1.224647e-16,-1.000000
3,2015-07-01,8953.0,73.18,37.804286,8901.0,166.41,27.68,2015,7,-5.000000e-01,-0.866025
4,2015-08-01,8535.0,72.58,63.944286,8531.0,167.12,27.85,2015,8,-8.660254e-01,-0.500000


In [22]:
df.shape

(129, 11)

In [23]:
df.isnull().sum()

Date                       0
Electricity_Requirement    0
Humidity                   0
Rainfall                   0
Electricity_Supply         0
Solar_Irradiance           0
Temperature                0
Year                       0
Month                      0
Month_sin                  0
Month_cos                  0
dtype: int64

# Lag Feature Cell

In [24]:
# Create lag features for Electricity Requirement

df["Demand_Lag_1"] = df["Electricity_Requirement"].shift(1)
df["Demand_Lag_2"] = df["Electricity_Requirement"].shift(2)
df["Demand_Lag_3"] = df["Electricity_Requirement"].shift(3)


# Remove rows with missing lag values
df = df.dropna().reset_index(drop=True)

print(df.head())

        Date  Electricity_Requirement  Humidity    Rainfall  \
0 2015-07-01                   8953.0     73.18   37.804286   
1 2015-08-01                   8535.0     72.58   63.944286   
2 2015-09-01                   8498.0     75.12  119.769714   
3 2015-10-01                   8330.0     78.86  153.807429   
4 2015-11-01                   6511.0     87.29  316.494857   

   Electricity_Supply  Solar_Irradiance  Temperature  Year  Month  Month_sin  \
0              8901.0            166.41        27.68  2015      7  -0.500000   
1              8531.0            167.12        27.85  2015      8  -0.866025   
2              8373.0            157.25        27.50  2015      9  -1.000000   
3              8324.0            143.69        26.60  2015     10  -0.866025   
4              6508.0             92.62        25.04  2015     11  -0.500000   

      Month_cos  Demand_Lag_1  Demand_Lag_2  Demand_Lag_3  
0 -8.660254e-01        8302.0        8381.0        8361.0  
1 -5.000000e-01     

In [25]:
print(df["Month"].unique())

[ 7  8  9 10 11 12  1  2  3  4  5  6]


In [26]:
print(df["Month"].value_counts().sort_index())

Month
1     10
2     10
3     10
4     10
5     10
6     10
7     11
8     11
9     11
10    11
11    11
12    11
Name: count, dtype: int64


In [27]:
print(df[["Date", "Month"]].head(20))

         Date  Month
0  2015-07-01      7
1  2015-08-01      8
2  2015-09-01      9
3  2015-10-01     10
4  2015-11-01     11
5  2015-12-01     12
6  2016-01-01      1
7  2016-02-01      2
8  2016-03-01      3
9  2016-04-01      4
10 2016-05-01      5
11 2016-06-01      6
12 2016-07-01      7
13 2016-08-01      8
14 2016-09-01      9
15 2016-10-01     10
16 2016-11-01     11
17 2016-12-01     12
18 2017-01-01      1
19 2017-02-01      2


In [30]:
print(df.columns.tolist())

['Date', 'Electricity_Requirement', 'Humidity', 'Rainfall', 'Electricity_Supply', 'Solar_Irradiance', 'Temperature', 'Year', 'Month', 'Month_sin', 'Month_cos', 'Demand_Lag_1', 'Demand_Lag_2', 'Demand_Lag_3']


In [28]:
# select the target
y = df["Electricity_Requirement"]

# We does require those columns

In [31]:
X = df.drop(
    columns=[
        "Date",
        "Electricity_Requirement",
        "Month"
    ]
)

In [32]:
print(X.columns)

Index(['Humidity', 'Rainfall', 'Electricity_Supply', 'Solar_Irradiance',
       'Temperature', 'Year', 'Month_sin', 'Month_cos', 'Demand_Lag_1',
       'Demand_Lag_2', 'Demand_Lag_3'],
      dtype='object')


In [33]:
print("Features Shape :", X.shape)
print("Target Shape   :", y.shape)

Features Shape : (126, 11)
Target Shape   : (126,)


In [34]:
split = int(len(df) * 0.8)

X_train = X.iloc[:split]
X_test = X.iloc[split:]

y_train = y.iloc[:split]
y_test = y.iloc[split:]

In [35]:
print("X_train :", X_train.shape)
print("X_test  :", X_test.shape)

print("y_train :", y_train.shape)
print("y_test  :", y_test.shape)

X_train : (100, 11)
X_test  : (26, 11)
y_train : (100,)
y_test  : (26,)


# saving the datasets

In [36]:
joblib.dump(X_train, "Demand_X_train.pkl")
joblib.dump(X_test, "Demand_X_test.pkl")

joblib.dump(y_train, "Demand_y_train.pkl")
joblib.dump(y_test, "Demand_y_test.pkl")

['Demand_y_test.pkl']

In [37]:
import os

for i in os.listdir():
    print(i)

00_Preprocessing.ipynb
01Randomforest.ipynb
02_XGBoost_Demand_Forecasting.ipynb
03_LightGBM_Demand_Forecasting.ipynb
Bidirectional_LSTM_Demand.ipynb
Demand_GRU_Baseline_08416.keras
Demand_LSTM_Baseline.keras
Demand_LSTM_Best_08356.keras
Demand_X_Scaler.pkl
Demand_X_test.pkl
Demand_X_train.pkl
Demand_y_Scaler.pkl
Demand_y_test.pkl
Demand_y_train.pkl
GRU_Demand.ipynb
LSTM_Demand_Forecasting.ipynb
Simple_RNN.ipynb
